In [81]:
import torch
import numpy as np
from tqdm import tqdm
from collections import defaultdict 
from poincare import PoincareManifold          # tes fichiers locaux
from model import Distance_PE

In [82]:
# Modifier à chaque fois :
checkpoint = torch.load('models/poincare_hpo_0.7_300.pt', map_location='cpu', weights_only=False)

objects = checkpoint['objects']
node2id = checkpoint['node2id']
losses = checkpoint['losses']
norm_history = checkpoint['norm_history']
edges = checkpoint['edges']
data = checkpoint['data']
hp = checkpoint['hyperparams']

In [83]:
manifold = PoincareManifold()
model = Distance_PE(n=len(objects), dim=hp['dim'],
                       manifold=manifold, sparse=True)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

Distance_PE(
  (embeddings): Embedding(19389, 2, sparse=True)
)

In [90]:
W = model.weight.detach().cpu().numpy()   # (N, dim)
norms = np.linalg.norm(W, axis=1)

print(f"Modèle chargé — {len(objects)} nœuds | dim={hp['dim']} | "
      f"{len(losses)} epochs")
print(f"Norme moy={norms.mean():.4f} | max={norms.max():.4f}")

i_min = np.argmin(norms).item()
print(f"Index de la plus petite norme : {i_min}")
print(f"Valeur de la norme : {W[i_min]}")


Modèle chargé — 19389 nœuds | dim=2 | 300 epochs
Norme moy=0.8056 | max=0.9711
Index de la plus petite norme : 1627
Valeur de la norme : [-0.0003496  0.0028319]


In [85]:
degrees = np.array([len(data.pos_neighbors[i]) for i in range(len(objects))])
norms = model.weight.detach().norm(dim=-1).numpy()

# Corrélation degré/norme attendue : négative
from scipy.stats import spearmanr
rho, pval = spearmanr(degrees, norms)
print(f"Corrélation Spearman degré/norme : {rho:.3f} (p={pval:.2e})")

Corrélation Spearman degré/norme : 0.066 (p=2.92e-20)


In [93]:
pos_neighbors = defaultdict(set)
pos_parents = defaultdict(set)

for u, v in edges:
    pos_neighbors[int(u)].add(int(v))
    pos_parents[int(v)].add(int(u))
    

len(pos_parents[i_min])
len(pos_parents[0])

0

In [95]:
degrees = np.array([len(data.pos_neighbors[i]) for i in range(len(objects))])

# Top 10 plus proches du centre
center_ids = np.argsort(norms)[:10]
print("=== 10 nœuds les plus proches du CENTRE ===")
for i in center_ids:
    print(f"  {objects[i]:<30} norme={norms[i]:.4f}  degré={degrees[i]}")

# Top 10 plus proches du bord
border_ids = np.argsort(norms)[-10:]
print("\n=== 10 nœuds les plus proches du BORD ===")
for i in border_ids:
    print(f"  {objects[i]:<30} norme={norms[i]:.4f}  degré={degrees[i]}")

=== 10 nœuds les plus proches du CENTRE ===
  HP:0002218                     norme=0.0029  degré=0
  HP:0011364                     norme=0.0031  degré=0
  HP:0011855                     norme=0.0052  degré=0
  HP:0002211                     norme=0.0071  degré=0
  HP:0002286                     norme=0.0084  degré=0
  HP:0011365                     norme=0.0099  degré=2
  HP:0002290                     norme=0.0100  degré=0
  HP:0005599                     norme=0.0107  degré=3
  HP:0011358                     norme=0.0108  degré=3
  HP:0007308                     norme=0.0407  degré=0

=== 10 nœuds les plus proches du BORD ===
  HP:0009769                     norme=0.9708  degré=8
  HP:0005918                     norme=0.9708  degré=26
  HP:0009407                     norme=0.9708  degré=3
  HP:0009845                     norme=0.9708  degré=5
  HP:0009832                     norme=0.9708  degré=16
  HP:0009858                     norme=0.9708  degré=5
  HP:0009774                   

In [79]:
from sklearn.metrics import average_precision_score

def evaluate(model, objects, edges, node2id):
    model.eval()
    W  = model.weight.detach()   # (N, dim)
 
    pos_neighbors = defaultdict(set)
    for u, v in edges:
        pos_neighbors[int(u)].add(int(v))

    ranksum, ap_scores = 0, 0
    nranks = 0
    iters = 0
    labels = np.empty(model.embeddings.weight.size(0))
 
    for u in tqdm(objects):
        labels.fill(0)
        u = int(node2id[u])
        neighbors = pos_neighbors.get(u, set())
        if not neighbors :
            continue
        u_exp = W[u].unsqueeze(0).expand(W.shape[0], -1)  # Coorconnées de u dans la boule de Poincaré
        dists = manifold.distance(u_exp, W).numpy()  # Distance de u aux autres noeuds
        dists[u] = 1e12
        #order = np.argsort(dists)  # Tri par distance décroissante p/r à u
        sorted_ind = np.argsort(dists)

        #ranks = int(np.where(order == v)[0][0]) + 1  # Rang du noeud v p/r à u dans l'embedding
        #ranks.append(rank)
        ranks, = np.where(np.isin(sorted_ind, list(neighbors)))
        ranks += 1
        N = ranks.shape[0]

        ranksum += ranks.sum() - (N * (N - 1) / 2)
        nranks += ranks.shape[0]
        labels[list(neighbors)] = 1
        ap_scores += average_precision_score(labels, -dists)
        iters += 1

        #pos  = pos_neighbors[u]  # Voisins de u dans la représentation initiale
        #hits, psum = 0, 0.0 
        #for k, idx in enumerate(order[1:], 1):  # On parcourt les noeuds du plus proche au plus éloigné
            #if idx in pos:
                #hits  += 1
                #psum  += hits / k
        #aps.append(psum / max(len(pos), 1))
 
    return float(ranksum), nranks, ap_scores, iters

In [88]:
results = evaluate(model, objects, edges, node2id)

print("Erreur moyenne sur le rang : ", float(results[0]) / results[1])
print("Erreur MAP moyenne :", float(results[2]) / results[3])

100%|██████████| 19389/19389 [00:39<00:00, 494.63it/s] 

Erreur moyenne sur le rang :  536.154242513832
Erreur MAP moyenne : 0.4784935480793084
